# Question 15: Crossing Frequency, Success Rate, and xG by League

**Tier 3 — Tactical and Opposition Analysis**

**Question:** How does crossing frequency and success rate vary by league, and is there a relationship between crossing volume and xG generated?

**Data:** StatsBomb Open Data + FBref

**Why this question matters:**  
The cross is one of the most debated actions in football analytics — widely used, rarely efficient at the team level, yet some teams build entire attacking identities around it. Understanding whether crossing volume actually generates xG, and whether that relationship varies by league, challenges a common coaching assumption and has direct implications for how wide players are recruited.

**What you are building toward:**
- Cross event analysis with outcome classification
- League-level comparison of crossing frequency and success
- Relationship between crossing volume and xG generated from crosses

---

**Critical lens (apply throughout):**
1. How does StatsBomb define a cross — and what deliveries does that exclude?
2. Is crossing success rate the right outcome measure — or is xG per cross more meaningful?
3. What team and player characteristics predict high-value crossing output?

## Environment Setup

In [1]:
# !pip install statsbombpy mplsoccer pandas numpy matplotlib seaborn




In [2]:
# Import libraries
from statsbombpy import sb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.optimize import minimize
import itertools
import mplsoccer

In [3]:
pd.set_option('display.max_columns', None)

## Step 1: Load cross events

Filter StatsBomb pass events to crosses. Identify outcome flags: successful delivery, blocked, out of play.

In [4]:
# Find a competition
competitions = sb.competitions()
competitions['competition_name'].unique()

world_cup_comp = competitions[competitions['competition_name'] == 'FIFA World Cup']
wc_2022 = world_cup_comp[world_cup_comp['season_name'] == '2022']


# Get matches for the 2022 World Cup
matches_2022 = sb.matches(competition_id=wc_2022['competition_id'].values[0], season_id=wc_2022['season_id'].values[0])

# Get lineups for all matches
# lineups_2022 = sb.lineups(competition_id=wc_2022['competition_id'].values[0], season_id=wc_2022['season_id'].values[0])

# Get events for all matches
match_id_list = matches_2022['match_id'].unique().tolist()
events = []
for match_id in match_id_list:
    match_events = sb.events(match_id=match_id)
    events.append(match_events)

events_2022 = pd.concat(events, ignore_index=True)



In [5]:
crossing_events_2022 = events_2022[events_2022['pass_cross'] == True][['pass_cross', 'location', 'pass_end_location', 'pass_outcome']]

## Step 2: Calculate crossing metrics by team and league

Crosses per 90, cross completion rate, xG generated from cross sequences per team.

In [6]:
# Crosses per 90, cross completion rate, xG generated from cross sequences per team

# Crosses per 90
crosses_per_90 = events_2022[events_2022['pass_cross'] == True].groupby('team').size() / (matches_2022.groupby('home_team').size() + matches_2022.groupby('away_team').size()) * 90
crosses_per_90 = crosses_per_90.reset_index().rename(columns={0: 'crosses_per_90'})
crosses_per_90.sort_values('crosses_per_90', ascending=False).reset_index(drop=True).head(10)


,team,crosses_per_90
0,Croatia,1491.428571
1,Mexico,1440.000000
2,Germany,1410.000000
3,Brazil,1404.000000
4,Iran,1260.000000
5,Serbia,1260.000000
6,United States,1237.500000
7,Spain,1237.500000
8,France,1208.571429
9,Denmark,1200.000000


In [7]:
# Cross completion rate
# Pass Outcome = NaN means a successful pass, so we can fill those with 'Complete' for the purposes of this calculation
events_2022['pass_outcome'] = events_2022['pass_outcome'].fillna('Complete')
cross_completion_rate = events_2022[events_2022['pass_cross'] == True].groupby('team')['pass_outcome'].apply(lambda x: ((x == 'Complete').sum() / x.count()) * 100)
cross_completion_rate


cross_completion_rate = cross_completion_rate.reset_index().rename(columns={'pass_outcome': 'cross_completion_rate'})
cross_completion_rate.sort_values('cross_completion_rate', ascending=False).reset_index(drop=True).head(10)

,team,cross_completion_rate
0,Poland,53.125000
1,Ecuador,46.428571
2,Qatar,43.333333
3,France,42.553191
4,Canada,40.540541
5,Senegal,38.636364
6,Germany,38.297872
7,Wales,35.714286
8,Belgium,33.333333
9,Uruguay,33.333333


In [8]:
events_2022

,50_50,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_won_advantage,foul_won_defensive,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure,block_offensive,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_penalty,goalkeeper_body_part,goalkeeper_shot_saved_to_post,pass_cut_back,shot_deflected,shot_saved_to_post,ball_recovery_offensive,pass_no_touch,pass_straight,shot_redirect,goalkeeper_punched_out,bad_behaviour_card,block_save_block,goalkeeper_shot_saved_off_target,shot_saved_off_target,goalkeeper_success_in_play,shot_follows_dribble,half_start_late_video_start,goalkeeper_lost_in_play
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1f60fe1e-0340-49e2-975e-f16f54c5da1a,1,NaN,NaN,NaN,3857276,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Canada,1833,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 4411, 'lineup': [{'player': {'id...",Canada,1833,00:00:00.000,Starting XI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d5ffdab0-a19a-497e-a0f6-d5403bfcc2b5,2,NaN,NaN,NaN,3857276,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Canada,1833,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 433, 'lineup': [{'player': {'id'...",Morocco,788,00:00:00.000,Starting XI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,c6dcab53-5a07-4a6e-ab8b-64e9f9133d19,3,NaN,NaN,NaN,3857276,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Canada,1833,[396c6ebb-5ba7-415c-ba37-88ff51f7c737],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Morocco,788,00:00:00.000,Half Start,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,396c6ebb-5ba7-415c-ba37-88ff51f7c737,4,NaN,NaN,NaN,3857276,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Canada,1833,[c6dcab53-5a07-4a6e-ab8b-64e9f9133d19],0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Can

In [9]:
# xG Generated from cross sequences per team
# First, we need to identify sequences that start with a cross and end with a shot.
# We can do this by iterating through the events and looking for patterns of a cross followed by a shot, with any number of events in between, within a certain time frame (e.g., 10 seconds).
def is_cross_to_shot_sequence(events, cross_index):
    # Check if the event at cross_index is a cross
    if not events.loc[cross_index, 'pass_cross']:
        return False
    
    # Look for a shot within the next 10 seconds
    cross_time = events.loc[cross_index, 'timestamp']
    for i in range(cross_index + 1, len(events)):
        if events.loc[i, 'type'] == 'Shot':
            shot_time = events.loc[i, 'timestamp']
            if (shot_time - cross_time).total_seconds() <= 10:
                return True
            else:
                return False
    return False

# Now we can iterate through the events and identify cross-to-shot sequences, summing the xG of the shots that result from crosses for each team.
cross_to_shot_xg = defaultdict(float)
for team in events_2022['team'].unique():
    team_events = events_2022[events_2022['team'] == team].reset_index(drop=True)
    for i in range(len(team_events)):
        if is_cross_to_shot_sequence(team_events, i):
            shot_xg = team_events.loc[i + 1, 'shot_statsbomb_xg']  # Assuming the shot is the next event after the cross
            cross_to_shot_xg[team] += shot_xg
cross_to_shot_xg_df = pd.DataFrame(list(cross_to_shot_xg.items()), columns=['team', 'cross_to_shot_xg'])
cross_to_shot_xg_df.sort_values('cross_to_shot_xg', ascending=False).reset_index(drop=True).head(10)


TypeError: unsupported operand type(s) for -: 'str' and 'str'

## Step 3: Compare across leagues

Do certain leagues use crosses significantly more or less? Is there a league-level crossing culture?

## Step 4: Scatter: crossing volume vs xG from crosses

Is there a diminishing returns effect — do high-volume crossing teams generate proportionally less xG per cross?

## Step 5: Player-level crossing quality

Which wide players generate the highest xG per cross? How does that compare to their raw cross volume ranking?

## Step 6: Cross delivery zones

Build a heatmap of cross origin zones vs xG generated. Which crossing zones are most dangerous?

## Interpretation and Critique

**1. Is there a positive relationship between crossing volume and xG generated — or does the data suggest diminishing returns?**

*(Write your answer here)*

**2. Which league has the most efficient crossing culture — and what tactical factors might explain it?**

*(Write your answer here)*

**3. A manager asks you whether to recruit a high-volume crosser or a low-volume but high-quality crosser. What does your data say?**

*(Write your answer here)*

**4. What does crossing analysis miss about how wide players create value?**

*(Write your answer here)*

---

## Before You Move On

1. Does your analysis pass the smell test — do the results make intuitive football sense?
2. What is the single biggest limitation of your methodology?
3. What new question did this analysis raise that it cannot answer?

Add new questions to `OPEN_QUESTIONS.md` before moving on.